# 08 — Hyperparameter Tuning, Threshold Calibration, and Evaluation

## Objective

Select model configurations using the designated development data, calibrate anomaly-score thresholds without using the final test labels, and evaluate the finalized models on the held-out test data.

This notebook separates three activities:

1. **Hyperparameter tuning** — select model configurations using the designated development data.
2. **Threshold calibration** — determine anomaly-detection thresholds from normal development data.
3. **Final evaluation** — apply the selected models and fixed thresholds to the held-out evaluation data.

The final test/attack labels are not used for hyperparameter selection or threshold calibration.

## Evaluation Principle

An anomaly-detection model produces a numerical score that represents how unusual an observation or sequence is. A threshold converts this score into a binary prediction.

Let $s_i$ denote the anomaly score for observation or sequence $i$, and let $\tau$ denote the selected threshold. The predicted label is

$$
\hat{y}_i =
\begin{cases}
1, & s_i > \tau,\\
0, & s_i \leq \tau,
\end{cases}
$$

where $\hat{y}_i = 1$ indicates a detected anomaly and $\hat{y}_i = 0$ indicates no detected anomaly.

### Reconstruction Error

For an autoencoder, reconstruction error measures the difference between the input and its reconstructed output.

Let $X_i \in \mathbb{R}^{L \times F}$ be input sequence $i$, and let $\hat{X}_i \in \mathbb{R}^{L \times F}$ be its reconstruction, where:

- $L$ is the sequence length;
- $F$ is the number of input features;
- $x_{i,t,f}$ is the observed value at timestep $t$ and feature $f$;
- $\hat{x}_{i,t,f}$ is the corresponding reconstructed value.

The mean squared reconstruction error for sequence $i$ is

$$
s_i =
\frac{1}{L F}
\sum_{t=1}^{L}
\sum_{f=1}^{F}
\left( x_{i,t,f} - \hat{x}_{i,t,f} \right)^2 .
$$

A larger value indicates a greater reconstruction discrepancy. However, a large reconstruction error is not, by itself, proof of an attack.

### Important Implementation Distinction

The equation above defines a sequence-level reconstruction score. It does not establish that every model in this project uses the same scoring procedure.

Before evaluating the models, the implementation must specify whether each method produces scores per observation, per timestep, or per sequence. For sliding windows, the relationship between sequence scores and the original timestamps must also be defined to ensure that predictions and ground-truth labels are aligned correctly.

Thresholds must be calibrated using the designated normal development data and fixed before final evaluation. Final test labels must not be used to select thresholds or tune model configurations.

## Data Separation

The project preserves the chronological structure of the SWaT time series.

Data used for:

- model training,
- hyperparameter selection,
- threshold calibration,
- and final evaluation

must remain explicitly separated.

In particular, final evaluation labels are not used to select hyperparameters or thresholds. This prevents information from the final evaluation set from influencing model decisions.

## Model Scope

The evaluation covers the anomaly-detection methods defined for this project:

### Baselines

- Rule-Based Detection
- Rolling Statistical Detection
- Simple Statistical Threshold

### Traditional Machine Learning

- Isolation Forest
- One-Class SVM
- Local Outlier Factor in novelty-detection mode

### Deep Learning

- LSTM Autoencoder
- TCN Autoencoder
- USAD

The preliminary TCN and USAD training results from Notebook 07 are treated as experimental evidence, not as final model-selection results.

## Evaluation Metrics

Performance will be assessed using appropriate classification and detection metrics, including:

- Precision
- Recall
- F1-score
- Confusion matrix
- False-positive and false-negative counts

Where appropriate, event-level behavior will also be examined in the subsequent error and event analysis stage.

No single metric will be interpreted in isolation.

In [ ]:
from pathlib import Path

# Locate the project root using the existing project structure.
CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        directory
        for directory in (CURRENT_DIR, *CURRENT_DIR.parents)
        if (directory / "data" / "raw").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root. "
        "Expected a directory containing 'data/raw'. "
        "Run this notebook from within the SWaT project directory."
    )

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

print(f"Raw data directory exists: {DATA_RAW_DIR.is_dir()}")
print(f"Notebooks directory exists: {NOTEBOOKS_DIR.is_dir()}")

Raw data directory exists: True
Notebooks directory exists: True


In [7]:
# Verify model artifacts in their actual project locations.

MODEL_DIR = PROJECT_ROOT / "models"

EXPECTED_ARTIFACTS = {
    "Isolation Forest": [
        MODEL_DIR / "isolation_forest.joblib",
    ],
    "One-Class SVM": [
        MODEL_DIR / "sgd_one_class_svm.joblib",
    ],
    "Local Outlier Factor": [
        MODEL_DIR / "lof_model.joblib",
    ],
    "LSTM Autoencoder": [
        MODEL_DIR / "lstm_autoencoder.pth",
    ],
    "TCN Autoencoder": [
        MODEL_DIR / "tcn_preliminary" / "tcn_autoencoder_best.pth",
    ],
    "USAD": [
        MODEL_DIR / "usad_preliminary" / "usad_best.pth",
    ],
    "Preprocessing": [
        MODEL_DIR / "scaler.joblib",
    ],
    "Evaluation sequences": [
        MODEL_DIR / "evaluation_sequences.npz",
    ],
}

print("NOTEBOOK 08 — MODEL ARTIFACT VERIFICATION")
print("=" * 48)

for model_name, paths in EXPECTED_ARTIFACTS.items():
    print(f"\n{model_name}:")

    for path in paths:
        if path.is_file():
            print(f"  [FOUND] {path.relative_to(PROJECT_ROOT)}")
        else:
            print(f"  [NOT FOUND] {path.relative_to(PROJECT_ROOT)}")

NOTEBOOK 08 — MODEL ARTIFACT VERIFICATION

Isolation Forest:
  [FOUND] models\isolation_forest.joblib

One-Class SVM:
  [FOUND] models\sgd_one_class_svm.joblib

Local Outlier Factor:
  [FOUND] models\lof_model.joblib

LSTM Autoencoder:
  [FOUND] models\lstm_autoencoder.pth

TCN Autoencoder:
  [FOUND] models\tcn_preliminary\tcn_autoencoder_best.pth

USAD:
  [FOUND] models\usad_preliminary\usad_best.pth

Preprocessing:
  [FOUND] models\scaler.joblib

Evaluation sequences:
  [FOUND] models\evaluation_sequences.npz


In [9]:
# Load the existing evaluation arrays once for use in Notebook 08.

import numpy as np

EVALUATION_FILE = PROJECT_ROOT / "models" / "evaluation_sequences.npz"

if not EVALUATION_FILE.is_file():
    raise FileNotFoundError(
        f"Evaluation file not found: {EVALUATION_FILE}"
    )

with np.load(EVALUATION_FILE, allow_pickle=False) as archive:
    EVALUATION_ARRAYS = {
        name: archive[name]
        for name in archive.files
    }

print("EVALUATION DATA READY")
print("=" * 40)

for name, array in EVALUATION_ARRAYS.items():
    print(
        f"{name}: "
        f"shape={array.shape}, "
        f"dtype={array.dtype}"
    )

EVALUATION DATA READY
X_threshold_seq: shape=(47408, 30, 42), dtype=float32
X_validation_seq: shape=(79031, 30, 42), dtype=float32
threshold_end_indices: shape=(47408,), dtype=int64
validation_end_indices: shape=(79031,), dtype=int64


In [10]:
# Prepare the existing evaluation arrays for subsequent model scoring.

X_threshold_seq = EVALUATION_ARRAYS["X_threshold_seq"]
X_validation_seq = EVALUATION_ARRAYS["X_validation_seq"]

threshold_end_indices = EVALUATION_ARRAYS["threshold_end_indices"]
validation_end_indices = EVALUATION_ARRAYS["validation_end_indices"]

# Validate sequence dimensions and index alignment.
assert X_threshold_seq.ndim == 3
assert X_validation_seq.ndim == 3

assert X_threshold_seq.shape[1:] == (30, 42)
assert X_validation_seq.shape[1:] == (30, 42)

assert len(X_threshold_seq) == len(threshold_end_indices)
assert len(X_validation_seq) == len(validation_end_indices)

print("Evaluation arrays prepared successfully.")
print(f"Threshold sequences: {X_threshold_seq.shape}")
print(f"Validation sequences: {X_validation_seq.shape}")
print(f"Threshold index range: {threshold_end_indices.min()}–{threshold_end_indices.max()}")
print(f"Validation index range: {validation_end_indices.min()}–{validation_end_indices.max()}")

Evaluation arrays prepared successfully.
Threshold sequences: (47408, 30, 42)
Validation sequences: (79031, 30, 42)
Threshold index range: 29–47436
Validation index range: 29–79059


In [ ]:
# Use the feature order recorded by the fitted scaler.
FEATURE_COLUMNS = list(scaler.feature_names_in_)

if len(FEATURE_COLUMNS) != 42:
    raise ValueError(
        f"Expected 42 features, found {len(FEATURE_COLUMNS)}."
    )

print("Traditional ML artifacts loaded successfully.")
print(f"Feature count: {len(FEATURE_COLUMNS)}")